# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Muhammadfaheem8988/FlyRank-AI-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### 1. Feature Vector Construction

* **Cohort Extraction**: Filtered the FlyRank data warehouse `fact_content_daily_performance` table for March 2026 (`month=2026-03`), selecting content items with active Search Console tracking (`gsc_data_available IS TRUE`) and a minimum search demand threshold of $\ge 10$ impressions.
* **Feature Vector Composition**: Constructed 8 pre-cutoff signals for 143,206 content items:
  1. `clicks`: Total March organic search clicks.
  2. `impressions`: Total March search impressions.
  3. `avg_position`: Impression-weighted mean search rank on Google.
  4. `ctr`: Realized click-through rate (`clicks / impressions`).
  5. `organic_sessions`: Total GA4 organic landing sessions.
  6. `avg_engagement_sec`: Average GA4 user engagement time per session.
  7. `active_days`: Distinct calendar days with search impressions in March.
  8. `ctr_gap`: Deficit between expected position CTR and realized CTR.
* **Categoricals & Imputation**: Client domain identifiers are cryptographically hashed (`client_hash_id`) and held out for validation grouping. Missing GA4 sessions or zero-click denominators are safely imputed with `0.0`.

In [1]:
import os
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN', '')

if hf_token:
    con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
    print("DuckDB Hugging Face secret configured.")
else:
    print("Warning: HF_TOKEN not found.")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
FACT_DAILY = f"{WAREHOUSE}/fact_content_daily_performance"

# Extract pre-cutoff features (March 2026) and future verification target (April 2026)
df = con.execute(f"""
    WITH march_obs AS (
        SELECT
            content_hash_id,
            ANY_VALUE(client_hash_id) AS client_hash_id,
            SUM(gsc_clicks) AS clicks,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position,
            SUM(gsc_clicks)::FLOAT / NULLIF(SUM(gsc_impressions), 0) AS ctr,
            SUM(sessions_organic) AS organic_sessions,
            SUM(ga4_total_engagement_sec)::FLOAT / NULLIF(SUM(sessions_organic), 0) AS avg_engagement_sec,
            COUNT(DISTINCT report_date) AS active_days
        FROM read_parquet('{FACT_DAILY}/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) >= 10 IS TRUE
    ),
    april_outcome AS (
        SELECT
            content_hash_id,
            SUM(gsc_clicks) AS april_clicks
        FROM read_parquet('{FACT_DAILY}/month=2026-04/*.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
    )
    SELECT
        m.*,
        COALESCE(a.april_clicks, 0) AS april_clicks,
        CASE WHEN COALESCE(a.april_clicks, 0) > m.clicks THEN 1 ELSE 0 END AS target_rebound
    FROM march_obs m
    LEFT JOIN april_outcome a ON m.content_hash_id = a.content_hash_id
""").df().fillna(0)

# Expected CTR curve for CTR gap feature
def expected_ctr(pos):
    if pos <= 3: return 0.20
    elif pos <= 6: return 0.08
    elif pos <= 10: return 0.03
    elif pos <= 20: return 0.015
    else: return 0.005

df['expected_ctr'] = df['avg_position'].apply(expected_ctr)
df['ctr_gap'] = (df['expected_ctr'] - df['ctr']).clip(lower=0)

features = [
    'clicks',
    'impressions',
    'avg_position',
    'ctr',
    'organic_sessions',
    'avg_engagement_sec',
    'active_days',
    'ctr_gap'
]

print("--- Feature Matrix Built ---")
print(f"Total Rows: {len(df):,}")
print(f"Features ({len(features)}): {features}")
display(df[['content_hash_id'] + features + ['target_rebound']].head(3))

DuckDB Hugging Face secret configured.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- Feature Matrix Built ---
Total Rows: 143,206
Features (8): ['clicks', 'impressions', 'avg_position', 'ctr', 'organic_sessions', 'avg_engagement_sec', 'active_days', 'ctr_gap']


,content_hash_id,clicks,impressions,avg_position,ctr,organic_sessions,avg_engagement_sec,active_days,ctr_gap,target_rebound
0,content_35d979572550dd7f,1.0,1423.0,16.168658,0.000703,0.0,0.0,31,0.014297,0
1,content_d95e1739253d6a65,0.0,208.0,25.264423,0.000000,0.0,0.0,27,0.005000,0
2,content_405cc2e0475690e9,0.0,607.0,30.583196,0.000000,0.0,0.0,31,0.005000,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### 2. Feature Inventory, Completeness, and Availability

| Feature | Data Type | Null Count | Min / Mean / Max | Operational Meaning & Missing Handling | Available When? |
|---|---|---|---|---|---|
| `clicks` | float64 | 0 | 0.0 / 5.73 / 5,668.0 | Total GSC organic clicks across March 2026. Non-null (imputed with 0). | Pre-cutoff (March 31) |
| `impressions` | float64 | 0 | 10.0 / 1,959.0 / 617,124.0 | Total March search impressions. Filtered $\ge 10$ to remove noise. | Pre-cutoff (March 31) |
| `avg_position` | float64 | 0 | 0.0 / 16.41 / 175.1 | Impression-weighted average ranking on Google SERP. | Pre-cutoff (March 31) |
| `ctr` | float32 | 0 | 0.0 / 0.0029 / 0.375 | Realized CTR (`clicks / impressions`). Zero clicks yield 0.0. | Pre-cutoff (March 31) |
| `organic_sessions` | float64 | 0 | 0.0 / 4.04 / 2,925.0 | Total landing sessions recorded in GA4. Missing/untracked filled with 0. | Pre-cutoff (March 31) |
| `avg_engagement_sec` | float32 | 0 | 0.0 / 2.50 / 1,917.0 | Mean GA4 engagement duration per organic session. Missing filled with 0. | Pre-cutoff (March 31) |
| `active_days` | int64 | 0 | 1.0 / 24.62 / 31.0 | Distinct days with impressions in March (measure of visibility consistency). | Pre-cutoff (March 31) |
| `ctr_gap` | float64 | 0 | 0.0 / 0.0444 / 0.200 | Deficit between benchmark SERP CTR and realized CTR. Lower bounded at 0. | Pre-cutoff (March 31) |

* **Zero Missingness**: All 143,206 rows carry 0 nulls across the 8 engineered features due to explicit SQL aggregation and zero-imputation.
* **Temporal Integrity**: Every feature vector value is established on or before midnight March 31, 2026, strictly preceding the April 2026 prediction and evaluation horizon.

In [2]:
# Check feature summary statistics, missingness, and data types
feature_profile = pd.DataFrame({
    'Data Type': df[features].dtypes,
    'Null Count': df[features].isnull().sum(),
    'Min': df[features].min().round(4),
    'Mean': df[features].mean().round(4),
    'Max': df[features].max().round(4)
})

print("--- Feature Vector Profile & Completeness (n=143,206) ---")
display(feature_profile)

--- Feature Vector Profile & Completeness (n=143,206) ---


,Data Type,Null Count,Min,Mean,Max
clicks,float64,0,0.0,5.7312,5668.000
impressions,float64,0,10.0,1959.0164,617124.000
avg_position,float64,0,0.0,16.4098,175.100
ctr,float32,0,0.0,0.0029,0.375
organic_sessions,float64,0,0.0,4.0355,2925.000
avg_engagement_sec,float32,0,0.0,2.4984,1917.000
active_days,int64,0,1.0,24.6176,31.000
ctr_gap,float64,0,0.0,0.0444,0.200


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### 3. Leakage Stress-Test and Split Isolation

#### Target & Future Window Leakage Test
* **Correlation Ceiling**: Correlating all 8 pre-cutoff signals against `target_rebound` reveals realistic, non-leaking correlations ranging from `-0.1161` (`avg_position`) to `+0.0919` (`impressions`).
* **Absence of Perfect Predictors**: No feature demonstrates suspiciously high correlation ($\ge 0.50$), confirming that neither post-cutoff April volumes (`april_clicks`, correlation `0.1807`) nor future recovery indicators contaminated the March feature extraction window.

#### Client & Domain Overlap Leakage
* **Partition Audit**: Evaluated a `GroupShuffleSplit` across `client_hash_id` allocating 33 domains to training and 12 domains to testing.
* **Zero Client Overlap**: Exactly **0 client domains overlap** between the training and test cohorts. This strictly prevents the model from memorizing domain-level authority, sitewide backlink profiles, or brand recognition during evaluation.

In [3]:
from sklearn.model_selection import GroupShuffleSplit

# 1. Check direct correlations with target_rebound to detect near-1.0 target leakage
correlations = df[features + ['april_clicks', 'target_rebound']].corr()['target_rebound'].sort_values(ascending=False)

print("--- Correlation with Ground Truth Target (target_rebound) ---")
display(correlations.to_frame(name='Correlation with Target').round(4))

# 2. Check for Client / Domain Leakage via GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_hash_id']))

train_clients = set(df.iloc[train_idx]['client_hash_id'].unique())
test_clients = set(df.iloc[test_idx]['client_hash_id'].unique())
client_overlap = train_clients.intersection(test_clients)

print("\n--- Domain & Client Split Leakage Check ---")
print(f"Train Client Count: {len(train_clients)}")
print(f"Test Client Count:  {len(test_clients)}")
print(f"Client Overlap Between Train & Test: {len(client_overlap)} (Must be 0)")

--- Correlation with Ground Truth Target (target_rebound) ---


,Correlation with Target
target_rebound,1.0000
april_clicks,0.1807
impressions,0.0919
clicks,0.0693
active_days,0.0687
organic_sessions,0.0654
ctr_gap,0.0594
avg_engagement_sec,0.0252
ctr,0.0056
avg_position,-0.1161



--- Domain & Client Split Leakage Check ---
Train Client Count: 33
Test Client Count:  12
Client Overlap Between Train & Test: 0 (Must be 0)


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### 4. Intentional Column & Cohort Exclusions

To preserve strict public safety, prevent target leakage, and eliminate statistical noise, the following variables and cohorts were intentionally excluded from model input:

#### 1. Excluded Raw Identifiers & Sensitive Fields
* **`page_url` / Plaintext URLs**: Anonymized to cryptographic hashes (`content_hash_id`) to ensure no proprietary client paths or sensitive topic hierarchies are exposed.
* **`client_name` / Domain Roots**: Mapped strictly to `client_hash_id` to comply with privacy agreements and prevent memorization of brand names.
* **`query` / Search Terms**: Raw search queries were excluded to eliminate high-cardinality sparse noise and protect user-level search privacy.

#### 2. Excluded Future & Target-Derived Fields
* **`april_clicks` / Q2 Performance Metrics**: Strictly withheld from the feature matrix; used solely as an out-of-sample ground truth label (`target_rebound`) during verification.

#### 3. Cohort Filters
* **Low-Impression URLs (`impressions < 10`)**: URLs receiving fewer than 10 impressions across March 2026 were discarded to prune crawl errors, unranked thin pages, and ephemeral indexing anomalies.
* **Inactive Tracking (`gsc_data_available IS FALSE`)**: Pages lacking active Search Console reporting were purged to avoid attributing measurement gaps to actual organic traffic declines.

In [4]:
# Verify excluded data and public safety
excluded_summary = {
    "excluded_columns": [
        "page_url / raw_url (anonymized to content_hash_id to protect client identity)",
        "query / search_term (excluded to prevent user privacy leaks and high-cardinality noise)",
        "client_name (anonymized to client_hash_id to satisfy NDA and privacy standards)",
        "april_clicks / future_metrics (strictly excluded from feature vector to prevent target leakage)"
    ],
    "excluded_cohorts": [
        "URLs with March impressions < 10 (removed unindexed noise and crawl artifacts)",
        "Rows where gsc_data_available is False (removed unmonitored domains)"
    ],
    "total_clean_rows": len(df),
    "unique_anonymized_clients": df['client_hash_id'].nunique()
}

import json
print("--- Data Exclusion & Public-Safety Audit ---")
print(json.dumps(excluded_summary, indent=2))

--- Data Exclusion & Public-Safety Audit ---
{
  "excluded_columns": [
    "page_url / raw_url (anonymized to content_hash_id to protect client identity)",
    "query / search_term (excluded to prevent user privacy leaks and high-cardinality noise)",
    "client_name (anonymized to client_hash_id to satisfy NDA and privacy standards)",
    "april_clicks / future_metrics (strictly excluded from feature vector to prevent target leakage)"
  ],
  "excluded_cohorts": [
    "URLs with March impressions < 10 (removed unindexed noise and crawl artifacts)",
    "Rows where gsc_data_available is False (removed unmonitored domains)"
  ],
  "total_clean_rows": 143206,
  "unique_anonymized_clients": 45
}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.